# Publicação dos anúncios no Pub/Sub

> Este notebook é a versão avulsa da seção 6 do [`gcp-pubsub-v2.ipynb`](gcp-pubsub-v2.ipynb), útil em um
> JupyterLab onde os dois arquivos convivem no mesmo diretório, como o do Dataproc. Nos notebooks do
> BigQuery Studio, onde cada notebook é um envio separado, use a seção 6 do notebook principal, que já
> traz estas mesmas células.

Ele lê um dos arquivos de anúncios coletados do Chaves na Mão e publica cada linha como uma mensagem no
tópico `aula-pdm-anuncios`.

Antes de executar, confira que os dados já foram descompactados:

```bash
unzip dados/anuncios.zip -d dados
```

In [ ]:
import google.auth

from google.api_core.exceptions import AlreadyExists
from google.cloud import pubsub_v1
from google.cloud.pubsub_v1.types import BatchSettings

from pathlib import Path

In [ ]:
_, project_id = google.auth.default()
print(project_id)

## 1. Escolha do arquivo

Dentro do diretório `dados/anuncios` existem 162 arquivos, cada um com os anúncios de um estado ou
de uma cidade. Você pode trocar o arquivo abaixo por qualquer outro — mas comece por um pequeno,
porque os arquivos de estado têm dezenas de milhares de anúncios.

O caminho é relativo à raiz do repositório, que é de onde o Jupyter abre este notebook.

In [ ]:
topic_id = "aula-pdm-anuncios"
anuncios_file = Path("dados/anuncios/imoveis_go-goiania.jsonl")

if not anuncios_file.exists():
    raise FileNotFoundError(
        f"{anuncios_file} não encontrado. "
        "Rode antes a célula de configuração do gcp-pubsub-v2.ipynb, que baixa os dados."
    )

anuncios_file

## 2. Publicação em lotes

Publicar uma mensagem por vez custa uma chamada de rede por anúncio. O cliente do Pub/Sub sabe agrupar
mensagens em lotes, e o `BatchSettings` define quando um lote é fechado: o que acontecer primeiro entre
os três limites abaixo.

In [ ]:
batch_settings = BatchSettings(
    max_bytes=5 * 1024 * 1024,  # No máximo 5MB por lote
    max_messages=1000,  # até 1000 messages por lote
    max_latency=0.25,  # envia a cada 250ms
)
publisher = pubsub_v1.PublisherClient(batch_settings=batch_settings)

In [ ]:
topic_path = publisher.topic_path(project_id, topic_id)
try:
    publisher.create_topic(name=topic_path)
except AlreadyExists:
    print(f"O tópico {topic_id} já existe.")

## 3. Leitura do arquivo

Cada linha do arquivo `.jsonl` já é um JSON completo, então não precisamos converter nada:
basta publicar a linha como ela está.

In [ ]:
# Lê o JSONL e armazena os JSONs convertidos em uma lista
linhas_json = anuncios_file.open("r", encoding="utf-8").readlines()
num_rows = len(linhas_json)

print(f"Foram lidas {num_rows} linhas do arquivo {anuncios_file}.")

In [ ]:
print(linhas_json[0])

## 4. Envio das mensagens

O método `publish` é assíncrono e devolve um `Future` — ele não espera a confirmação do servidor.
Por isso a célula abaixo termina quase que instantaneamente, mesmo com milhares de anúncios: o envio
de verdade acontece em segundo plano.

O terceiro argumento (`row`) vira um atributo da mensagem. Atributos são metadados que viajam junto
com o corpo e podem ser usados para filtrar mensagens na assinatura.

In [ ]:
futures = []
for idx, linha in enumerate(linhas_json):
    future = publisher.publish(topic_path, data=linha.encode("utf-8"), row=str(idx))
    futures.append(future)

É o `result()` que espera a confirmação de cada publicação. Se alguma mensagem falhar,
é aqui que a exceção aparece.

In [ ]:
for fut in futures:
    fut.result(timeout=60)

print(f"{len(futures)} anúncios publicados no tópico {topic_id}.")